# 00C2_qwen2_5_vl_7b_real_model_two_item_smoke.ipynb

Generated zero-edit Kaggle runbook. Attach the documented private datasets, keep Internet off, choose the documented accelerator, and click Run All. NON_EVIDENCE_RUNTIME_SMOKE; paper_evidence=false.


In [ ]:
# Generated immutable run identity. There is nothing to edit in this notebook.
import os

STAGE = 'real_model_smoke'
PROVIDER = 'qwen2_5_vl_7b'
NOTEBOOK_NAME = '00C2_qwen2_5_vl_7b_real_model_two_item_smoke.ipynb'
EXPECTED_GPUS = 2
ALLOW_SINGLE_GPU_FALLBACK = True
USE_REAL_MODEL = True
MAX_ITEMS = 2
GLOBAL_SEED = 12013
SCHEMA_VERSION = "certvic.cvpr.output.v2"
SNAPSHOT_CONTRACT = "UNIFIED_SNAPSHOT"
PROMPT_TEMPLATE_ID = "certification_yes_no_v1"
PROMPT_TEMPLATE = "{prompt}\n"
PARSER_VERSION = "certvic.parse.v2"
CANONICAL_RETURN_ZIP = '00C2_qwen2_5_vl_7b_real_model_smoke.zip'
LOCAL_DESTINATION = 'data/runtime/00C2_qwen2_5_vl_7b_real_model_smoke.zip'
INPUT_ROOT = os.environ.get("CERTVIC_KAGGLE_INPUT_ROOT", "/kaggle/input")
WORKING_ROOT = os.environ.get("CERTVIC_KAGGLE_WORKING_ROOT", "/kaggle/working")
SNAPSHOT_DATASET_SLUG = 'certvic/qwen2-5-vl-7b-snapshot'
SNAPSHOT_DATASET_FILENAME = 'qwen2_5_vl_7b_snapshot.zip'
for key, value in {
    "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "DIFFUSERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1", "HF_HUB_DISABLE_TELEMETRY": "1", "PIP_NO_INDEX": "1",
    "PIP_DISABLE_PIP_VERSION_CHECK": "1",
}.items():
    os.environ[key] = value


In [ ]:
import hashlib, json, pathlib, shutil, stat, subprocess, sys, zipfile

EARLY_ERRORS = {
    "missing": "KAGGLE_BOOTSTRAP_01_DATASET_NOT_FOUND",
    "ambiguous": "KAGGLE_BOOTSTRAP_02_AMBIGUOUS_DATASET",
    "invalid": "KAGGLE_BOOTSTRAP_03_BUNDLE_INVALID",
    "unsafe": "KAGGLE_BOOTSTRAP_09_UNSAFE_EXTRACTION",
}

def early_sha256(path):
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def early_member(info):
    name = info.filename
    normalized = name.replace("\\", "/")
    value = pathlib.PurePosixPath(normalized)
    mode = (info.external_attr >> 16) & 0xFFFF
    if (not normalized or normalized != name or normalized.endswith("/") or value.is_absolute()
            or ".." in value.parts or "." in value.parts or normalized.startswith("~")
            or "\x00" in normalized or info.is_dir() or stat.S_ISLNK(mode)
            or (mode and not stat.S_ISREG(mode))):
        raise RuntimeError(f"{EARLY_ERRORS['unsafe']}: unsafe member {name!r}")
    return value.as_posix()

def early_verify_and_extract(archive_path, destination):
    source = pathlib.Path(archive_path).resolve()
    try:
        with zipfile.ZipFile(source) as archive:
            infos = archive.infolist()
            names = [early_member(info) for info in infos]
            if len(names) != len(set(names)) or archive.testzip() is not None:
                raise RuntimeError(f"{EARLY_ERRORS['invalid']}: duplicate or corrupt members")
            manifest = json.loads(archive.read("bundle_manifest.json"))
            hashes = json.loads(archive.read("hash_manifest.json"))
            if (manifest.get("schema") != "certvic.kaggle.bundle.v1"
                    or hashes.get("schema") != "certvic.kaggle.hash_manifest.v1"
                    or manifest.get("bundle_type") != 'CODE'
                    or manifest.get("expected_kaggle_dataset_slug") != 'certvic/certvic-code'):
                raise RuntimeError(f"{EARLY_ERRORS['invalid']}: code bundle identity mismatch")
            declared = manifest.get("files", {})
            hash_files = hashes.get("files", {})
            if (set(names) != set(hash_files) | {"hash_manifest.json"}
                    or set(declared) != set(names) - {"bundle_manifest.json", "hash_manifest.json"}):
                raise RuntimeError(f"{EARLY_ERRORS['invalid']}: code file universe mismatch")
            for name, record in hash_files.items():
                payload = archive.read(name)
                observed = {"size": len(payload), "sha256": hashlib.sha256(payload).hexdigest()}
                if record != observed or (name in declared and declared[name] != observed):
                    raise RuntimeError(f"{EARLY_ERRORS['invalid']}: byte mismatch {name}")
            target = pathlib.Path(destination).resolve()
            if target.exists():
                if target.is_symlink() or not target.is_dir():
                    raise RuntimeError(f"{EARLY_ERRORS['unsafe']}: invalid destination")
                shutil.rmtree(target)
            target.mkdir(parents=True)
            for info, name in zip(infos, names, strict=True):
                output = (target / name).resolve()
                try:
                    output.relative_to(target)
                except ValueError as error:
                    raise RuntimeError(f"{EARLY_ERRORS['unsafe']}: traversal member") from error
                output.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as reader, output.open("xb") as writer:
                    shutil.copyfileobj(reader, writer, length=1024 * 1024)
    except (OSError, KeyError, json.JSONDecodeError, zipfile.BadZipFile) as error:
        raise RuntimeError(f"{EARLY_ERRORS['invalid']}: {error}") from error
    return target, manifest

mount = pathlib.Path(INPUT_ROOT) / 'certvic-code'
matches = sorted(path.resolve() for path in mount.rglob('certvic_code_bundle.zip') if path.is_file()) if mount.is_dir() else []
if not matches:
    raise RuntimeError(f"{EARLY_ERRORS['missing']}: slug=certvic/certvic-code filename=certvic_code_bundle.zip")
if len(matches) != 1:
    raise RuntimeError(f"{EARLY_ERRORS['ambiguous']}: slug=certvic/certvic-code candidates={[str(p) for p in matches]}")
CODE_BUNDLE_PATH = str(matches[0])
CODE_BUNDLE_HASH = early_sha256(matches[0])
CODE_EXTRACT_ROOT, CODE_OUTER_MANIFEST = early_verify_and_extract(
    matches[0], pathlib.Path(WORKING_ROOT) / "certvic_code"
)
project_candidates = sorted(path.parent.resolve() for path in CODE_EXTRACT_ROOT.rglob("pyproject.toml")
    if (path.parent / "certvic/__init__.py").is_file())
if len(project_candidates) != 1:
    raise RuntimeError("KAGGLE_BOOTSTRAP_10_AMBIGUOUS_CONTENT: project root")
PROJECT_ROOT = project_candidates[0]
sys.path.insert(0, str(PROJECT_ROOT))

from certvic.cvpr.notebook_bootstrap import (
    discover_unique_file, discover_unique_root, materialize_dataset, verify_attached_bundle,
)
verify_attached_bundle(CODE_BUNDLE_PATH, expected_type='CODE', expected_slug='certvic/certvic-code')
print({"project_root": str(PROJECT_ROOT), "code_bundle_sha256": CODE_BUNDLE_HASH})


In [ ]:
from certvic.cvpr.environment_lock import environment_lock_hash

CONFIG_DATASET = materialize_dataset(
    slug='certvic/certvic-configs', filename='certvic_configs_bundle.zip', expected_type='CONFIGS',
    input_root=INPUT_ROOT, destination=pathlib.Path(WORKING_ROOT) / "certvic_configs",
)
TOOLS_DATASET = materialize_dataset(
    slug='certvic/certvic-execution-tools', filename='certvic_execution_tools_bundle.zip', expected_type='EXECUTION_TOOLS',
    input_root=INPUT_ROOT, destination=pathlib.Path(WORKING_ROOT) / "certvic_execution_tools",
)
WHEELHOUSE_DATASET = materialize_dataset(
    slug='certvic/certvic-offline-wheelhouse', filename='certvic_offline_wheelhouse.zip', expected_type='OFFLINE_LINUX_WHEELHOUSE',
    input_root=INPUT_ROOT, destination=pathlib.Path(WORKING_ROOT) / "certvic_offline_wheelhouse",
)
CONFIG_ROOT = pathlib.Path(CONFIG_DATASET["root"])
WHEELHOUSE_ROOT = pathlib.Path(WHEELHOUSE_DATASET["root"])
ENVIRONMENT_LOCK = str(discover_unique_file(CONFIG_ROOT, "kaggle_t4x2_environment.lock.json"))
ENVIRONMENT_LOCK_HASH = environment_lock_hash(ENVIRONMENT_LOCK)
WHEELHOUSE_MANIFEST = str(discover_unique_file(WHEELHOUSE_ROOT, "wheelhouse_manifest.json"))
WHEELHOUSE_PATH = str(WHEELHOUSE_ROOT / "wheels")
if not pathlib.Path(WHEELHOUSE_PATH).is_dir():
    raise RuntimeError("KAGGLE_BOOTSTRAP_04_WHEELHOUSE_INVALID: wheels directory missing")
MODEL_REGISTRY = str(discover_unique_file(CONFIG_ROOT, "certvic_immutable_model_registry.json"))
ATTACHED_INPUT_HASHES = {
    "code": CODE_BUNDLE_HASH,
    "configs": CONFIG_DATASET["archive_sha256"],
    "tools": TOOLS_DATASET["archive_sha256"],
    "wheelhouse": WHEELHOUSE_DATASET["archive_sha256"],
}
print({"environment_lock": ENVIRONMENT_LOCK, "environment_lock_hash": ENVIRONMENT_LOCK_HASH,
       "wheelhouse_manifest": WHEELHOUSE_MANIFEST, "attached_input_hashes": ATTACHED_INPUT_HASHES})


In [ ]:
from certvic.cvpr.model_snapshot_manifest import verify_manifest

SNAPSHOT_DATASET = materialize_dataset(
    slug='certvic/qwen2-5-vl-7b-snapshot', filename='qwen2_5_vl_7b_snapshot.zip', expected_type="MODEL_SNAPSHOT",
    input_root=INPUT_ROOT,
    destination=pathlib.Path(WORKING_ROOT) / 'certvic_snapshot_qwen2_5_vl_7b',
)
SNAPSHOT_CONTAINER = pathlib.Path(SNAPSHOT_DATASET["root"])
SNAPSHOT_MANIFEST = str(discover_unique_file(
    SNAPSHOT_CONTAINER, "certvic_model_snapshot_manifest.json"
))
SNAPSHOT_ROOT = pathlib.Path(SNAPSHOT_MANIFEST).parent.resolve()
MODEL_PATH = str(SNAPSHOT_ROOT)
PROCESSOR_PATH = str(SNAPSHOT_ROOT)
SNAPSHOT_MANIFEST_HASH = early_sha256(SNAPSHOT_MANIFEST)
snapshot_identity = json.loads(pathlib.Path(SNAPSHOT_MANIFEST).read_text(encoding="utf-8"))
MODEL_ID = str(snapshot_identity["model_id"])
PROCESSOR_ID = str(snapshot_identity["processor_id"])
MODEL_COMMIT = str(snapshot_identity["model_commit"])
PROCESSOR_COMMIT = str(snapshot_identity["processor_commit"])
EXPECTED_ARCHITECTURE = str(snapshot_identity["expected_architecture"])
SNAPSHOT_ROOT_HASH = str(snapshot_identity["unified_snapshot_root_sha256"])
outer_snapshot = SNAPSHOT_DATASET["bundle_manifest"]
for field, expected in {
    "provider": PROVIDER, "model_id": MODEL_ID, "model_commit": MODEL_COMMIT,
    "processor_commit": PROCESSOR_COMMIT, "expected_architecture": EXPECTED_ARCHITECTURE,
    "unified_snapshot_root_sha256": SNAPSHOT_ROOT_HASH,
}.items():
    if outer_snapshot.get(field) != expected:
        raise RuntimeError(f"KAGGLE_BOOTSTRAP_03_BUNDLE_INVALID: snapshot {field} mismatch")
registry = json.loads(pathlib.Path(MODEL_REGISTRY).read_text(encoding="utf-8"))["models"][PROVIDER]
if (registry.get("repository_id") != MODEL_ID
        or registry.get("model_commit") != MODEL_COMMIT
        or registry.get("processor_commit") != PROCESSOR_COMMIT
        or registry.get("architecture") != EXPECTED_ARCHITECTURE):
    raise RuntimeError("KAGGLE_BOOTSTRAP_08_RUN_IDENTITY_INCOMPLETE: immutable registry mismatch")
ATTACHED_INPUT_HASHES["snapshot"] = SNAPSHOT_DATASET["archive_sha256"]
print({"snapshot_root": MODEL_PATH, "snapshot_manifest": SNAPSHOT_MANIFEST,
       "snapshot_manifest_sha256": SNAPSHOT_MANIFEST_HASH,
       "snapshot_root_sha256": SNAPSHOT_ROOT_HASH, "model_id": MODEL_ID,
       "model_commit": MODEL_COMMIT, "processor_commit": PROCESSOR_COMMIT,
       "expected_architecture": EXPECTED_ARCHITECTURE})


In [ ]:
from certvic.cvpr.contracts import canonical_json_bytes, sha256_bytes
from certvic.cvpr.notebook_permission_binding import derive_permission_binding
from certvic.cvpr.reconcile_provider_permissions import (
    provider_state, transition_provider_permission, verify_matrix_authorization,
    verify_provider_permission,
)
from certvic.cvpr.run_contract import build_run_contract
from certvic.cvpr.task_bundle import verify_bundle as verify_task_bundle

SMOKE_DATASET = materialize_dataset(
    slug="certvic/certvic-real-two-item-smoke",
    filename="certvic_real_two_item_smoke_bundle.zip",
    expected_type="REAL_TWO_ITEM_SMOKE_INPUT",
    input_root=INPUT_ROOT,
    destination=pathlib.Path(WORKING_ROOT) / "certvic_real_two_item_smoke",
)
PERMISSION_DATASET = materialize_dataset(
    slug="certvic/certvic-pre-smoke-permissions",
    filename="certvic_pre_smoke_permissions.zip",
    expected_type="PRE_SMOKE_PERMISSIONS",
    input_root=INPUT_ROOT,
    destination=pathlib.Path(WORKING_ROOT) / "certvic_pre_smoke_permissions",
)
SMOKE_ROOT = pathlib.Path(SMOKE_DATASET["root"])
PERMISSION_ROOT = pathlib.Path(PERMISSION_DATASET["root"])
TASK_BUNDLE_MANIFEST = str(discover_unique_file(SMOKE_ROOT, "task_bundle_manifest.json"))
TASK_BUNDLE_ROOT = str(pathlib.Path(TASK_BUNDLE_MANIFEST).parent)
bundle_verification = verify_task_bundle(TASK_BUNDLE_ROOT, TASK_BUNDLE_MANIFEST)
TASK_BUNDLE_HASH = str(bundle_verification["bundle_hash"])
TASK_MANIFEST = str(pathlib.Path(bundle_verification["tasks_path"]).resolve())
active_tasks = [json.loads(line) for line in pathlib.Path(TASK_MANIFEST).read_text(
    encoding="utf-8").splitlines() if line]
if len(active_tasks) != 2:
    raise RuntimeError("KAGGLE_ZERO_EDIT_00C2_TASK_CARDINALITY_INVALID")

def unique_schema_file(root, schema, *, provider=None):
    matches = []
    for path in pathlib.Path(root).rglob("*.json"):
        if not path.is_file() or path.is_symlink():
            continue
        try:
            value = json.loads(path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError, UnicodeDecodeError):
            continue
        if value.get("schema") == schema and (provider is None or value.get("provider") == provider):
            matches.append(path.resolve())
    if len(matches) != 1:
        raise RuntimeError(
            f"KAGGLE_ZERO_EDIT_00C2_PERMISSION_ARTIFACT_INVALID: schema={schema} provider={provider} "
            f"matches={[str(path) for path in matches]}"
        )
    return matches[0]

MATRIX_AUTHORIZATION = str(unique_schema_file(
    PERMISSION_ROOT, "certvic.cvpr.matrix_authorization.v1"
))
PROVIDER_PERMISSION = str(unique_schema_file(
    PERMISSION_ROOT, "certvic.cvpr.provider_permission.v1", provider=PROVIDER
))
matrix_authorization = verify_matrix_authorization(MATRIX_AUTHORIZATION)
permission = verify_provider_permission(
    PROVIDER_PERMISSION, matrix=MATRIX_AUTHORIZATION, expected_provider=PROVIDER
)
if permission.get("runtime_class") != "REAL_MODEL_SMOKE":
    raise RuntimeError("KAGGLE_ZERO_EDIT_00C2_PERMISSION_CLASS_INVALID")

known_role_paths = {
    "task_bundle_manifest": pathlib.Path(TASK_BUNDLE_MANIFEST),
    "environment_lock": pathlib.Path(ENVIRONMENT_LOCK),
    "model_registry": pathlib.Path(MODEL_REGISTRY),
    "snapshot_manifest": pathlib.Path(SNAPSHOT_MANIFEST),
    "code_bundle": pathlib.Path(CODE_BUNDLE_PATH),
    "matrix_authorization": pathlib.Path(MATRIX_AUTHORIZATION),
}
search_roots = [PERMISSION_ROOT, SMOKE_ROOT, CONFIG_ROOT]
resolved_roles = {}
for role, expected_hash in sorted(permission["active_input_hashes"].items()):
    preferred = known_role_paths.get(role)
    if preferred is not None and preferred.is_file() and early_sha256(preferred) == expected_hash:
        matches = [preferred.resolve()]
    else:
        matches = sorted({
            path.resolve() for root in search_roots for path in root.rglob("*")
            if path.is_file() and not path.is_symlink() and early_sha256(path) == expected_hash
        })
    if len(matches) != 1:
        raise RuntimeError(
            f"KAGGLE_ZERO_EDIT_00C2_PERMISSION_BINDING_MISSING: role={role} "
            f"matches={[str(path) for path in matches]}"
        )
    resolved_roles[role] = str(matches[0])

required_roles = {
    "task_bundle_manifest", "freeze_manifest", "final_review", "smoke_gate",
    "environment_lock", "model_registry", "snapshot_manifest", "code_bundle",
    "study_config", "matrix_authorization",
}
if set(resolved_roles) != required_roles:
    raise RuntimeError("KAGGLE_ZERO_EDIT_00C2_PERMISSION_ROLE_MATRIX_INCOMPLETE")
FINAL_TASK_FREEZE = resolved_roles["freeze_manifest"]
FINAL_REVIEW_LEDGER = resolved_roles["final_review"]
SMOKE_GATE_JSON = resolved_roles["smoke_gate"]
STUDY_CONFIG = resolved_roles["study_config"]
RUN_TAG = str(permission["active_scalars"]["run_tag"])
STUDY = str(permission["study"])
if permission["active_scalars"].get("provider") != PROVIDER:
    raise RuntimeError("KAGGLE_ZERO_EDIT_00C2_PROVIDER_SCALAR_MISMATCH")
if permission["active_scalars"].get("schema_version") != SCHEMA_VERSION:
    raise RuntimeError("KAGGLE_ZERO_EDIT_00C2_SCHEMA_SCALAR_MISMATCH")
PROMPT_TEMPLATE_HASH = hashlib.sha256(PROMPT_TEMPLATE.encode("utf-8")).hexdigest()
if permission.get("prompt_template_hash") != PROMPT_TEMPLATE_HASH:
    raise RuntimeError("KAGGLE_ZERO_EDIT_00C2_PROMPT_HASH_MISMATCH")

permission_binding = derive_permission_binding(globals())
active_runtime_contract_input = {
    "study": STUDY, "runtime_class": "REAL_MODEL_SMOKE", "provider": PROVIDER,
    "model_id": MODEL_ID, "processor_id": PROCESSOR_ID,
    "model_commit": MODEL_COMMIT, "processor_commit": PROCESSOR_COMMIT,
    "model_snapshot_manifest_hash": SNAPSHOT_MANIFEST_HASH,
    "processor_snapshot_manifest_hash": SNAPSHOT_MANIFEST_HASH,
    "snapshot_status": "LOCAL_SNAPSHOT_BYTES_VERIFIED",
    "snapshot_contract": SNAPSHOT_CONTRACT,
    "environment_lock_hash": ENVIRONMENT_LOCK_HASH,
    "prompt_template_id": PROMPT_TEMPLATE_ID,
    "prompt_template_hash": PROMPT_TEMPLATE_HASH,
    "parser_version": PARSER_VERSION, "output_schema": SCHEMA_VERSION,
    "run_tag": RUN_TAG, "code_bundle_hash": CODE_BUNDLE_HASH, "seed": GLOBAL_SEED,
    "generation_parameters": {"do_sample": False, "max_new_tokens": 8},
}
active_run_contract = build_run_contract(
    active_runtime_contract_input,
    task_manifest_sha256=sha256_bytes(canonical_json_bytes(active_tasks)), strict=True,
)
checks = {
    "active_input_hashes": permission["active_input_hashes"] == permission_binding["input_hashes"],
    "active_scalars": permission["active_scalars"] == permission_binding["scalars"],
    "task_bundle_hash": permission["task_bundle_hash"] == TASK_BUNDLE_HASH,
    "environment_hash": permission["environment_hash"] == ENVIRONMENT_LOCK_HASH,
    "snapshot_hash": permission["snapshot_hash"] == SNAPSHOT_MANIFEST_HASH,
    "snapshot_root_hash": permission["snapshot_root_hash"] == SNAPSHOT_ROOT_HASH,
    "code_hash": permission["code_hash"] == CODE_BUNDLE_HASH,
    "prompt_template_hash": permission["prompt_template_hash"] == PROMPT_TEMPLATE_HASH,
    "run_contract_hash": permission["run_contract_hash"] == active_run_contract["run_contract_hash"],
    "parser_version": permission["parser_version"] == PARSER_VERSION,
}
if not all(checks.values()):
    raise RuntimeError(f"KAGGLE_ZERO_EDIT_00C2_PERMISSION_IDENTITY_MISMATCH: {checks}")
PROVIDER_PERMISSION_EVENTS = str(
    pathlib.Path(WORKING_ROOT) / f"{PROVIDER}_permission_events.jsonl"
)
state = provider_state(PROVIDER_PERMISSION_EVENTS, permission)
if state == "ISSUED":
    transition_provider_permission(
        permission, PROVIDER_PERMISSION_EVENTS, to_state="CLAIMED",
        actor=NOTEBOOK_NAME, detail={"binding_hash": permission_binding["binding_hash"]},
    )
elif state not in {"CLAIMED", "RUN_STARTED", "PACKAGING_FAILED"}:
    raise RuntimeError(f"KAGGLE_ZERO_EDIT_00C2_PERMISSION_NOT_RESUMABLE: state={state}")
ATTACHED_INPUT_HASHES.update({
    "snapshot": SNAPSHOT_DATASET["archive_sha256"],
    "smoke": SMOKE_DATASET["archive_sha256"],
    "permissions": PERMISSION_DATASET["archive_sha256"],
})
print({"permission_id": permission["permission_id"], "run_tag": RUN_TAG,
       "task_bundle_hash": TASK_BUNDLE_HASH, "resolved_permission_roles": resolved_roles})


In [ ]:
from certvic.cvpr.environment_lock import (
    offline_environment_flags, prepare_offline_environment,
)
from certvic.cvpr.notebook_bootstrap import configure_offline_environment, import_smoke
from certvic.cvpr.runtime_preflight import hardware_report

configure_offline_environment()
if offline_environment_flags().get("HF_HUB_OFFLINE") != "1" or os.environ.get("PIP_NO_INDEX") != "1":
    raise RuntimeError("KAGGLE_ZERO_EDIT_OFFLINE_POLICY_INCOMPLETE")
environment_verification = prepare_offline_environment(
    ENVIRONMENT_LOCK,
    wheelhouse=WHEELHOUSE_PATH,
    wheelhouse_manifest=WHEELHOUSE_MANIFEST,
    allow_preinstalled=True,
    require_exact=True,
    require_cuda=True,
)
if environment_verification["status"] not in {
    "EXACT_PREINSTALLED_ENVIRONMENT_ACCEPTED", "OFFLINE_WHEELHOUSE_INSTALLED_AND_VERIFIED",
}:
    raise RuntimeError("KAGGLE_ZERO_EDIT_EXACT_ENVIRONMENT_NOT_ESTABLISHED")
hardware = hardware_report()
print(hardware)
if EXPECTED_GPUS == 0 and (hardware["cuda_available"] or hardware["gpu_count"] != 0):
    raise RuntimeError("KAGGLE_ZERO_EDIT_CPU_ACCELERATOR_MUST_BE_OFF")
if EXPECTED_GPUS > 0:
    names = [row["name"] for row in hardware.get("gpus", [])]
    if not hardware["cuda_available"]:
        raise RuntimeError("KAGGLE_BOOTSTRAP_07_GPU_CONTRACT_FAILED: CUDA unavailable")
    if len(names) < 2 and not (len(names) == 1 and ALLOW_SINGLE_GPU_FALLBACK):
        raise RuntimeError(f"KAGGLE_BOOTSTRAP_07_GPU_CONTRACT_FAILED: device_count={len(names)}")
    if not all("T4" in name.upper() for name in names[:2]):
        raise RuntimeError(f"KAGGLE_BOOTSTRAP_07_GPU_CONTRACT_FAILED: devices={names}")
import_versions = import_smoke(["certvic", "numpy", "pandas", "torch", "transformers"])
print({"offline": True, "network_used": False, "imports": import_versions,
       "environment_status": environment_verification["status"]})


In [ ]:
from certvic.cvpr.model_snapshot_manifest import verify_manifest
from certvic.cvpr.t4x2 import derive_seed_manifest, detect_topology, write_seed_manifest

snapshot = verify_manifest(
    MODEL_PATH, SNAPSHOT_MANIFEST,
    expected_model_id=MODEL_ID,
    expected_model_commit=MODEL_COMMIT,
    expected_processor_commit=PROCESSOR_COMMIT,
    expected_architecture=EXPECTED_ARCHITECTURE,
)
if not snapshot["passed"]:
    raise RuntimeError(f"KAGGLE_ZERO_EDIT_SNAPSHOT_INVALID: {snapshot['errors']}")
device_names = [row["name"] for row in hardware.get("gpus", [])]
T4_PLAN = detect_topology(
    device_names=device_names, allow_single_t4=ALLOW_SINGLE_GPU_FALLBACK
)
print(T4_PLAN.as_dict())

OUTPUT_DIR = str(pathlib.Path(WORKING_ROOT) / f"certvic_00c2_{PROVIDER}")
RUNTIME_CONFIG = str(pathlib.Path(WORKING_ROOT) / f"00C2_{PROVIDER}_runtime.json")
runtime = {
    **active_runtime_contract_input,
    "model_path": MODEL_PATH, "processor_path": PROCESSOR_PATH,
    "snapshot_root_hash": SNAPSHOT_ROOT_HASH,
    "snapshot_manifest_path": SNAPSHOT_MANIFEST,
    "expected_architecture": EXPECTED_ARCHITECTURE,
    "environment_lock_path": ENVIRONMENT_LOCK,
    "prompt_template": PROMPT_TEMPLATE,
    "strict_run_contract": True, "strict_permission_binding": True,
    "task_manifest": TASK_MANIFEST, "task_bundle_root": TASK_BUNDLE_ROOT,
    "task_bundle_manifest": TASK_BUNDLE_MANIFEST, "task_bundle_hash": TASK_BUNDLE_HASH,
    "output_dir": OUTPUT_DIR,
    "final_task_freeze": FINAL_TASK_FREEZE,
    "final_review_ledger": FINAL_REVIEW_LEDGER,
    "smoke_gate_json": SMOKE_GATE_JSON,
    "model_registry": MODEL_REGISTRY, "study_config": STUDY_CONFIG,
    "code_bundle": CODE_BUNDLE_PATH,
    "matrix_authorization": MATRIX_AUTHORIZATION,
    "execution_permission_id": permission["permission_id"],
    "execution_permission_signature": permission["content_signature_sha256"],
    "provider_permission_path": PROVIDER_PERMISSION,
    "provider_permission_events_path": PROVIDER_PERMISSION_EVENTS,
    "permission_binding": permission_binding,
    "notebook_name": NOTEBOOK_NAME,
    "canonical_smoke_destination": str(pathlib.Path(WORKING_ROOT) / CANONICAL_RETURN_ZIP),
    "defer_canonical_smoke_package": True,
}
output_root = pathlib.Path(OUTPUT_DIR)
output_root.mkdir(parents=True, exist_ok=True)
write_seed_manifest(output_root / "seed_manifest.json", derive_seed_manifest(
    global_seed=GLOBAL_SEED, study=STUDY, provider=PROVIDER, gpu_id=0, shard_id=0,
    task_ids=[str(row["item_id"]) for row in active_tasks], attempts=2,
))
(output_root / "environment_manifest.json").write_text(json.dumps({
    "schema": "certvic.cvpr.smoke_environment.v1",
    "environment_hash": ENVIRONMENT_LOCK_HASH,
    "environment_lock_hash": ENVIRONMENT_LOCK_HASH,
    "status": environment_verification["status"], "passed": True,
    "hardware": hardware, "network_used": False, "paper_evidence": False,
}, indent=2, sort_keys=True) + "\n", encoding="utf-8")
pathlib.Path(RUNTIME_CONFIG).write_text(
    json.dumps(runtime, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
command = [
    sys.executable, "-m", "certvic.cvpr.worker", "--shard", "0", "--num-shards", "1",
    "--resume", "--batch-size", "2", "--oom-reduce-to-one", "--fail-closed",
    "--frozen-runtime-config", RUNTIME_CONFIG,
]
subprocess.run(command, check=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
subprocess.run([
    sys.executable, "-m", "certvic.cvpr.package_run",
    "--frozen-runtime-config", RUNTIME_CONFIG, "--expected-shards", "1",
], check=True)
canonical = pathlib.Path(WORKING_ROOT) / CANONICAL_RETURN_ZIP
if not canonical.is_file():
    raise RuntimeError(f"KAGGLE_ZERO_EDIT_CANONICAL_RETURN_MISSING: {CANONICAL_RETURN_ZIP}")
print(str(canonical))
print(f"DOWNLOAD_FILENAME={CANONICAL_RETURN_ZIP}")
print(f"LOCAL_DESTINATION={LOCAL_DESTINATION}")
print("RESUME_COMMAND=python3 scripts/run_all_cpu_workflows.py --resume")
